# Stage 9B.0 / 9B.0.1 - Nominal F300 4F Virtual Bench and Candidate Atlas

This notebook is a nominal, unvalidated scalar 4F diagnostic only. Stage 9B.0.1 routes candidate fields through the existing CSLM component route before the nominal F300 model starts at the field arriving at SLM2. It does not make the physical 4F route ready, does not model pixelated-SLM order physics, does not model a camera, does not run inverse correction or AI, and does not introduce material response.

Boundary labels: `nominal_4f_forward_model`, `not_bench_calibrated`, `not_physical_4f_readiness_ready`, `not_camera_modelled`, `not_material_modelled`, `final_export_allowed=False`.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from vbb_study.digital_twin.nominal_f300_4f import (
    CLAIM_BOUNDARY_LABELS,
    NominalF300Config,
    config_from_profile,
    load_nominal_f300_profile,
    nominal_4f_sanity_report,
    plot_component_sequence,
    run_nominal_f300_4f,
    run_to_manifest,
    stop_sampling_report,
)
from vbb_study.digital_twin.candidate_beam_atlas import (
    DEFAULT_RUN_ID,
    build_candidate_ranking_validity_rows,
    build_candidate_specs,
    evaluate_candidate_stop_sampling_convergence,
    export_candidate_package,
    plot_candidate_atlas,
    plot_candidate_ranking_validity,
    plot_stop_robustness,
    plot_stop_sampling_convergence,
    plot_upstream_cslm_to_nominal_4f_chain,
    relay_output_candidate_metrics,
    simulate_candidate,
)


In [ ]:
profile = load_nominal_f300_profile()
study = json.loads(Path('configs/studies/cslm_nominal_4f_candidate_atlas_v1.json').read_text())
display(Markdown('## Nominal Profile'))
display(pd.DataFrame([
    {'parameter': key, 'value': entry['value'], 'provenance': entry['provenance']}
    for key, entry in profile['known_nominal_geometry'].items()
]))
display(Markdown('## Boundary'))
boundary = {label: True for label in CLAIM_BOUNDARY_LABELS + ('final_export_allowed_false',)}
boundary.update(profile.get('carrier_boundary', {}))
display(pd.Series(boundary))
display(Markdown('## Stop Sampling Policy'))
display(pd.DataFrame([profile['stop_sampling_policy']['exploratory_profile'], profile['stop_sampling_policy']['standard_profile']]))


In [ ]:
config = config_from_profile()
reference_spec = [spec for spec in build_candidate_specs() if spec.candidate_id == 'gaussian_reference'][0]
run = simulate_candidate(reference_spec, config)
manifest = run_to_manifest(run)
display(Markdown('## Executed Nominal Component Chain'))
display(pd.DataFrame(manifest['component_manifest']))
display(Markdown('## Upstream Bridge'))
display(pd.Series({
    'upstream_source_mode': run.upstream_source_mode,
    'slm1_phase_applied_at_slm1': run.slm1_phase_applied_at_slm1,
    'slm1_to_slm2_propagation_included': run.slm1_to_slm2_propagation_included,
    'carrier_realism': run.carrier_realism,
    'pixelated_slm_diffraction_orders_modelled': run.pixelated_slm_diffraction_orders_modelled,
}))
display(Markdown('## Energy Ledger'))
display(pd.DataFrame(manifest['component_energy_ledger']))
display(Markdown('## Sanity Report'))
display(pd.Series(nominal_4f_sanity_report(run)))


## Stage 9B.0.1 - Upstream CSLM Bridge and Stop-Sampling Validity

Candidate runs now use `existing_cslm_component_route` so SLM1 phase is applied at SLM1 and propagated to SLM2 before the nominal F300 relay. The SLM2 carrier is an ideal continuous-ramp surrogate, not pixelated-SLM diffraction-order physics. Ranking is only shown after stop sampling convergence passes.


In [ ]:
component_fig = plot_component_sequence(run)
stop_fig = plot_stop_robustness()
atlas_fig = plot_candidate_atlas()
bridge_fig = plot_upstream_cslm_to_nominal_4f_chain()
convergence_fig = plot_stop_sampling_convergence()
ranking_fig = plot_candidate_ranking_validity()
for fig in (component_fig, stop_fig, atlas_fig, bridge_fig, convergence_fig, ranking_fig):
    display(Image(filename=str(fig)))


In [ ]:
specs = build_candidate_specs()
ranking_rows = build_candidate_ranking_validity_rows(specs)
display(pd.DataFrame(ranking_rows))
display(Markdown('Only `gaussian_reference` and `vortex_ell_1` through `vortex_ell_4` are in the Stage 9B.0.1 initial shortlist.'))


In [ ]:
package_spec = [spec for spec in build_candidate_specs() if spec.candidate_id == 'vortex_ell_2'][0]
package_paths = export_candidate_package(package_spec, run_id=DEFAULT_RUN_ID)
display(Markdown('## Demonstrator Candidate Package'))
display(pd.Series({key: str(path) for key, path in package_paths.items()}))
claim_boundary_path = Path('outputs/nominal_4f_candidate_runs') / DEFAULT_RUN_ID / package_spec.candidate_id / 'claim_boundary.md'
display(Markdown(claim_boundary_path.read_text(encoding='utf-8')))


## Unsupported

Still unsupported: measured physical 4F coordinates, true stop radius/position, SLM phase response calibration, camera modelling, inverse correction, AI, relay-output-to-axicon handoff geometry, material response, and final export.